Goal is to construct a prox operator for use with ADMM with a diffusion model.
The required techniques can be readily found in this paper: [A VARIATIONAL PERSPECTIVE ON SOLVING INVERSE PROBLEMS WITH DIFFUSION MODELS](https://arxiv.org/pdf/2305.04391) 

In the original paper a gradient is proposed however we will build a stochastic prox by taking 
$$ || x_0 - x_i ||^2 $$ 
as additional objective.

In [1]:
using Ferrite
using ModularEIT
using Images
using IterativeSolvers
using LinearAlgebra
using Plots
using Distributions
using Statistics
using Lux
using JLD2

In [2]:
include("model/sde.jl")

#wrap_model##0 (generic function with 1 method)

In [3]:
η = 0.15

σf = 0.0e-2 # how much noise is added to f
σg = 0.0e-2 # How much noise is added to g
# Steers prox operators:
ρ_obj = 0.0

0.0

In [20]:
img = load("1096.jpg")
itp = interpolate_array_2D(Float64.(img))
n = 63 
grid = generate_grid(Quadrilateral, (n, n));
∂Ω = union(getfacetset.((grid,), ["left", "top", "right", "bottom"])...)
fe  = FerriteFESpace{RefQuadrilateral}(grid,2,3,∂Ω)
cond_vec = project_function_to_fem(fe, itp)
cond_vec .= min.(max.(cond_vec,1e-6),1.0)
itp = interpolate_array_2D(Float64.(img))
n = 63 
grid = generate_grid(Quadrilateral, (n, n));
∂Ω = union(getfacetset.((grid,), ["left", "top", "right", "bottom"])...)
fe  = FerriteFESpace{RefQuadrilateral}(grid,2,3,∂Ω)
cond_vec = project_function_to_fem(fe, itp)
cond_vec .= min.(max.(cond_vec,1e-6),1.0)
G_full = real_fourier_basis(8)
rhs_dict = Dict()
Threads.@threads for i in 2:256
    M = make_boundary(G_full[:, i],64)
    itp = interpolate_array_2D(M)
    rhs_dict[i] = assemble_rhs_func(fe, itp)
end
K = assemble_L(fe, cond_vec)
K_fac = cholesky(K)
mode_dict = Dict{Int64,FerriteEITMode}()
mode_dict_no_noise = Dict{Int64,FerriteEITMode}()
@time begin
    Threads.@threads for i in 2:256
        mode_dict[i-1] = create_mode_from_g(fe, rhs_dict[i], K_fac, normalize =  true, σ_f = σf, σ_g= σg)
        mode_dict_no_noise[i-1] = create_mode_from_g(fe, rhs_dict[i], K_fac, normalize =  true)
    end
end
img_start = load("Tikhonov/1.000e-01_0.000e+00_0.000e+00_0.000e+00.png")
itp_start = interpolate_array_2D(Float64.(img_start))
σ_vec = project_function_to_fem(fe, itp_start)
σ_vec .= min.(max.(σ_vec,1e-6),1.0)
using Distributions
#σ_vec = rand(Uniform(1e-6, 1.0), fe.n)
sol = FerriteSolverState(fe, σ_vec)
prblm = FerriteProblem(fe, mode_dict, sol)
eval_points = reshape(equidistant_grid(64), :)
ph = PointEvalHandler(grid, eval_points)
f,∂f = create_f∂f(prblm, 255; gn=true)
#prox_obj = create_prox_linesearch(f, ∂f, ρ_obj)
prox_obj = create_proximal_gradient_step(f, ∂f, ρ_obj, fe.n; ub=1.0)

  0.872197 seconds (88.38 k allocations: 826.167 MiB, 41.74% gc time, 4.35% compilation time)


#78 (generic function with 1 method)

In [21]:
function grad_A(x)
    # convert to Float 64 and shift:
    xf64 = Float64.(denormalize_image(x))
    itp64 = interpolate_array_2D(xf64)
    σ64 = project_function_to_fem(fe, itp64)
    ∂f(σ64) 
    img64 = reshape(evaluate_at_points(ph, prblm.fe.dh, ∂f(σ64)),(64,64))
    Float32.(normalize_image(img64))
end
function obj_A(x)
    # convert to Float 64 and shift:
    xf64 = Float64.(denormalize_image(x))
    itp64 = interpolate_array_2D(xf64)
    σ64 = project_function_to_fem(fe, itp64)
    f(σ64)
end

obj_A (generic function with 1 method)

In [22]:
function prox_A(x)
    # convert to Float 64 and shift:
    xf64 = Float64.(denormalize_image(x))
    itp64 = interpolate_array_2D(xf64)
    σ64 = project_function_to_fem(fe, itp64)
    x, err_x, p_x = prox_obj(σ64)
    img64 = reshape(evaluate_at_points(ph, prblm.fe.dh, x),(64,64))
    return Float32.(normalize_image(img64)), err_x, p_x
end

prox_A (generic function with 1 method)

In [23]:
function model_gradient(μ, model, λ, T)
    # sample t ∈ [0, T]
    t = rand(Uniform(0, T))
    αbar = ᾱ(t)

    # sample noise
    ε = randn(size(μ))

    # forward evaluate
    xt = sqrt(αbar) .* μ .+ sqrt(1 - αbar) .* ε
    ϵ_pred = model(xt, t)

    # λ_t = λ / SNR_t = λ * σ_t / α_t
    αt = sqrt(αbar)
    σt = sqrt(1 - αbar)
    λ_t = λ * σt / αt

    err = dot(ϵ_pred .- ε, μ)

    grad_diff = ϵ_pred .- ε 
    grad =  λ_t .* grad_diff

    return grad, t, err, λ
end

model_gradient (generic function with 1 method)

In [24]:
function create_SDE_prox(model, steps,λ,T = 1.0)
    prox = x₀ -> begin

            obj = 0
            xᵢ = copy(x₀)
            t_ges = 0
            for i in 1:steps
                grad, t, err, _ = model_gradient(xᵢ, model, λ, T)
                xᵢ += grad
                obj += err
                t_ges += t
            end
            #obj_diff = 0.5 * ρ sum(abs2, x .- x₀)
            #grad_diff = ρ .* (x .- x₀)
        return xᵢ, err, t_ges
    end
    return prox
end

create_SDE_prox (generic function with 2 methods)

In [25]:
prox_sde = create_SDE_prox(sde,10,0.0)

#create_SDE_prox##2 (generic function with 1 method)

In [26]:
σ₁ = reshape(evaluate_at_points(ph, prblm.fe.dh, σ_vec),(64,64))
σ₁ = Float32.(normalize_image(σ₁))

64×64 Matrix{Float32}:
  0.152941   0.152941   0.160784  …   0.160784   0.152941   0.160784
  0.152941   0.152941   0.160784      0.168628   0.160784   0.168628
  0.160784   0.160784   0.160784      0.168628   0.168628   0.176471
  0.176471   0.176471   0.176471      0.168628   0.168628   0.168628
  0.2        0.2        0.192157      0.160784   0.160784   0.160784
  0.231373   0.223529   0.215686  …   0.152941   0.145098   0.145098
  0.254902   0.247059   0.239216      0.137255   0.137255   0.129412
  0.278431   0.262745   0.254902      0.129412   0.121569   0.113726
  0.286275   0.278431   0.270588      0.121569   0.113726   0.0980393
  0.294118   0.286275   0.278431      0.113726   0.105882   0.105882
  ⋮                               ⋱                        
 -0.623529  -0.631373  -0.639216  …  -0.858824  -0.85098   -0.866667
 -0.670588  -0.670588  -0.686275     -0.866667  -0.858824  -0.87451
 -0.733333  -0.72549   -0.733333     -0.87451   -0.866667  -0.858824
 -0.788235  -0.78039

In [27]:
σᵢ,_ = prox_sde(σ₁)

([0.15294122695922852 0.15294122695922852 … 0.15294122695922852 0.16078436374664307; 0.15294122695922852 0.15294122695922852 … 0.16078436374664307 0.16862750053405762; … ; -0.8980392217636108 -0.8980392217636108 … -0.9137254953384399 -0.9137254953384399; -0.9058823585510254 -0.9058823585510254 … -0.9058823585510254 -0.9058823585510254], nothing, 4.495690392637103)

In [28]:
using Printf
function admm(prox_f, prox_g, x0; iters=10, print_err=false)
    fmt(x) = @sprintf("%.3e", x) 
    x = copy(x0)
    z = copy(x0)
    u = zero(x0)
    err_x = 0
    err_z = 0
    for k in 1:iters
        x, err_x, p_x = prox_f(z .- u)
        z, err_z, p_z = prox_g(x .+ u)
        u = u .+ x .- z
        if print_err
            println("iter $k: objective: $err_x")
        end
    end
    return x, z, u
end

admm (generic function with 1 method)

In [29]:
σⱼ , _ ,_ = prox_A(σ₁)
#for i in 1:10
#    σⱼ, err1 ,_ = prox_A(σⱼ)
#    println("step $i error: $err1")
#end

(Float32[0.15294123 0.15294123 … 0.15294123 0.16078436; 0.15294123 0.15294123 … 0.16078436 0.1686275; … ; -0.8980392 -0.8980392 … -0.9137255 -0.9137255; -0.90588236 -0.90588236 … -0.90588236 -0.90588236], 0.029318890886637982, 0.0)

In [30]:
σⱼ

64×64 Matrix{Float32}:
  0.152941   0.152941   0.160784  …   0.160784   0.152941   0.160784
  0.152941   0.152941   0.160784      0.168628   0.160784   0.168628
  0.160784   0.160784   0.160784      0.168628   0.168628   0.176471
  0.176471   0.176471   0.176471      0.168628   0.168628   0.168628
  0.2        0.2        0.192157      0.160784   0.160784   0.160784
  0.231373   0.223529   0.215686  …   0.152941   0.145098   0.145098
  0.254902   0.247059   0.239216      0.137255   0.137255   0.129412
  0.278431   0.262745   0.254902      0.129412   0.121569   0.113726
  0.286275   0.278431   0.270588      0.121569   0.113726   0.0980393
  0.294118   0.286275   0.278431      0.113726   0.105882   0.105882
  ⋮                               ⋱                        
 -0.623529  -0.631373  -0.639216  …  -0.858824  -0.85098   -0.866667
 -0.670588  -0.670588  -0.686275     -0.866667  -0.858824  -0.87451
 -0.733333  -0.72549   -0.733333     -0.87451   -0.866667  -0.858824
 -0.788235  -0.78039

In [ ]:
x,u,z = admm(prox_A, prox_sde, σᵢ ; print_err = true)

iter 1: objective: 0.029318890886637982
iter 2: objective: 0.029318890886637982


In [ ]:
solution = Gray.(clamp.(denormalize_image(σⱼ),0.0,1.0))

In [ ]:
img